# Assignment 5 — Many-Objective Optimisation with JUSTICE (Local Run)

**Course:** EPA141A Model-Based Decision Making — Delft University of Technology  
**Model:** JUSTICE  

---

## Learning Outcomes

After completing this assignment you will be able to:
1. Load and inspect an optimisation configuration from a JSON file produced in Assignment 4.
2. Run a many-objective evolutionary algorithm on the JUSTICE model via `run_optimization_local.py`.
3. Load and inspect the Pareto-front CSV files produced by a completed MOEA run.

---

## Background

Assignment 4 defined the optimisation problem: 244 RBF decision variables, four objectives, and epsilon values that control archive resolution. This assignment runs the actual Pareto search.

The optimiser is **GenerationalBorg**, a self-adaptive MOEA that simultaneously applies six variation operators (SBX, PCX, DE, UNDX, SPX, UM) and continuously adjusts their selection probabilities based on which generates the most archive improvement. This makes it well suited to high-dimensional spaces like the 244-parameter RBF search space; however, due to the use of many operators, may be slower.  Explore what other algorithms are available in the [ema_workbench optimization docs](https://emaworkbench.readthedocs.io/en/latest/ema_documentation/em_framework/optimization.html) and [Platypus](https://platypus.readthedocs.io/en/latest/api/platypus.algorithms.html), and consider whether a different algorithm is more appropriate for your problem. Any algorithm can be passed to the `algorithm=` argument in `run_optimization_local.py`.


Each function evaluation runs the JUSTICE model once with one candidate policy (one set of 244 RBF parameters), returning four objective values. The algorithm accumulates non-dominated solutions in an **epsilon-archive**: a candidate is only archived if it improves on the current archive by at least ε in at least one objective. Smaller ε gives finer Pareto resolution but requires more function evaluations.

Because MOEAs are stochastic, a single run may miss parts of the Pareto front. Running five independent seeds and pooling their archives with epsilon-dominance produces a **combined reference set** that is more complete than any individual seed. This reference set is the starting point for the Pareto and robustness analyses in Assignments 6 and 7.

---

## Starting Point — Your Assignment 4 configuration

In Assignment 4 you defined the multi-objective optimisation problem and saved your choices to `config/config_student.json`. This notebook loads that config and runs the optimisation using `run_optimization_local.py`, which first needs your input. 


---

## Step 1 — Inspect the Configuration

The optimisation is governed by `config/config_student.json`.  
Read and print each key with an explanation.

In [7]:
import warnings
warnings.filterwarnings("ignore")

import os, sys, json
import numpy as np
import pandas as pd

# Locate JUSTICE-main (one level up from model_answers_ema/)
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_JUSTICE_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "../JUSTICE-main"))

if _JUSTICE_ROOT not in sys.path:
    sys.path.insert(0, _JUSTICE_ROOT)

os.chdir(_JUSTICE_ROOT)

# ── Inspect config ────────────────────────────────────────────────────────────
CONFIG_PATH = os.path.join(_NOTEBOOK_DIR, "../config/config_student.json")

with open(CONFIG_PATH) as fh:
    cfg = json.load(fh)

explanations = {
    "start_year":                       "Simulation start year",
    "end_year":                         "Simulation end year",
    "data_timestep":                    "Years between raw input data points",
    "timestep":                         "Model integration timestep (years)",
    "emission_control_start_year":      "First year ECR can exceed zero",
    "n_rbfs":                           "Number of RBFs (effective: n_inputs + 2)",
    "n_inputs":                         "RBF input signals ",
    "epsilons":                         "Archive granularity",
    "temperature_year_of_interest":     "Year for threshold fraction evaluation",
    "reference_ssp_rcp_scenario_index": "Reference scenario index",
}

print(f"Configuration: {CONFIG_PATH}\n")
for k, v in cfg.items():
    print(f"  {k:<40s}  {str(v):<15}  # {explanations.get(k, '')}")

Configuration: c:\Users\marij\OneDrive\Documenten\Master-EPA\Model-Based-DM\epa141a_01\assignments_ema\../config/config_student.json

  start_year                                2015             # Simulation start year
  end_year                                  2300             # Simulation end year
  data_timestep                             5                # Years between raw input data points
  timestep                                  1                # Model integration timestep (years)
  emission_control_start_year               2025             # First year ECR can exceed zero
  n_rbfs                                    4                # Number of RBFs (effective: n_inputs + 2)
  n_inputs                                  2                # RBF input signals 
  epsilons                                  [10.0, 0.01, 10.0, 10.0]  # Archive granularity
  temperature_year_of_interest              2100             # Year for threshold fraction evaluation
  reference_ssp_rcp_scenari

In [8]:
from justice.util.data_loader import DataLoader

# Decision variable count
n_inputs   = cfg["n_inputs"]
n_rbfs_act = n_inputs + 2
n_regions  = len(DataLoader().REGION_LIST)

n_centers = n_rbfs_act * n_inputs
n_radii   = n_rbfs_act * n_inputs
n_weights = n_rbfs_act * n_regions
n_total   = n_centers + n_radii + n_weights

print("RBF decision variable summary:")
print(f"  n_rbfs_actual = {n_rbfs_act}  (n_inputs + 2 = {n_inputs} + 2)")
print(f"  n_regions     = {n_regions}")
print(f"  ─────────────────────────────────")
print(f"  Centers       = {n_rbfs_act} × {n_inputs} = {n_centers:4d}  range [-1,  1]")
print(f"  Radii         = {n_rbfs_act} × {n_inputs} = {n_radii:4d}  range [ 0,  1]")
print(f"  Weights       = {n_rbfs_act} × {n_regions} = {n_weights:4d}  range [ 0,  1]")
print(f"  ─────────────────────────────────")
print(f"  TOTAL                    = {n_total}")

print(f"\nEpsilon values per objective:")
obj_names = ["welfare", "fraction_above_threshold", "welfare_loss_damage", "welfare_loss_abatement"]
for name, eps in zip(obj_names, cfg["epsilons"]):
    print(f"  {name:<35s}  ε = {eps}")

RBF decision variable summary:
  n_rbfs_actual = 4  (n_inputs + 2 = 2 + 2)
  n_regions     = 57
  ─────────────────────────────────
  Centers       = 4 × 2 =    8  range [-1,  1]
  Radii         = 4 × 2 =    8  range [ 0,  1]
  Weights       = 4 × 57 =  228  range [ 0,  1]
  ─────────────────────────────────
  TOTAL                    = 244

Epsilon values per objective:
  welfare                              ε = 10.0
  fraction_above_threshold             ε = 0.01
  welfare_loss_damage                  ε = 10.0
  welfare_loss_abatement               ε = 10.0


---

## Step 2 — Run the Optimisation

The optimisation is implemented in `run_optimization_local.py`. Go through it
and make sure that your configuration is properly integrated!  Update it with your own modeling choices.  This is very important. You can run the current python code as is, to get a feel, but make the adjustments necessary 
for your particular problem formulation.
Open a **terminal** in the `assignments_ema/` directory and run:

```bash
# Quick smoke-test (~3-5 min, 1 seed, 500 NFE) highly suggested before a fully fledged run
python run_optimization_local.py --nfe 500 --seeds 9845531

# Full single-seed run (~1-3 h)
python run_optimization_local.py --nfe 50000 --seeds 9845531

# Full 5-seed run (background, ~1 working day)
nohup python run_optimization_local.py --nfe 50000 > opt_log.txt 2>&1 &
```

You can also launch it from the notebook cell below after you complete it (it blocks the kernel until finished, it may take a while):

In [3]:
# ── CONFIGURE HERE ───────────────────────────────────────────────────────────
NFE         = 2000 # YOUR NUMBER OF FUNCTION EVALUATIONS
SEEDS       = [1, 2, 3, 4, 5]  # YOUR CODE run at least 5 random seeds, eg. 1,2,3..
N_ENSEMBLES = 10               # YOUR CODE            #  FAIR members (local speed)
N_PROCESSES = None             # None → auto-detect (cpu_count - 1)
OUTPUT_DIR  = os.path.join(_NOTEBOOK_DIR, "results")
# ─────────────────────────────────────────────────────────────────────────────

seeds_str  = " ".join(str(s) for s in SEEDS)
n_proc_arg = f"--n_processes {N_PROCESSES}" if N_PROCESSES else ""

# Derive script path from _JUSTICE_ROOT (reliable regardless of kernel CWD)
script_path = os.path.normpath(
    os.path.join(_JUSTICE_ROOT, "..", "assignments_ema", "run_optimization_local.py")
)
cmd = (
    f"python {script_path} "
    f"--nfe {NFE} "
    f"--seeds {seeds_str} "
    f"--n_ensembles {N_ENSEMBLES} "
    f"--output_dir {OUTPUT_DIR} "
    f"--config " + os.path.join(_NOTEBOOK_DIR, "../config/config_student.json") + " "
    + n_proc_arg
)

print("Command to run:")
print(f"  {cmd}")
print("\nRunning ... (this cell blocks until complete)")

ret = os.system(cmd)
print(f"\nExit code: {ret}  ({'OK' if ret == 0 else 'ERROR'})")

Command to run:
  python c:\Users\marij\OneDrive\Documenten\Master-EPA\Model-Based-DM\epa141a_01\assignments_ema\run_optimization_local.py --nfe 2000 --seeds 1 2 3 4 5 --n_ensembles 10 --output_dir c:\Users\marij\OneDrive\Documenten\Master-EPA\Model-Based-DM\epa141a_01\assignments_ema\results --config c:\Users\marij\OneDrive\Documenten\Master-EPA\Model-Based-DM\epa141a_01\assignments_ema\../config/config_student.json 

Running ... (this cell blocks until complete)

Exit code: 0  (OK)


---

## Step 3 — Load and Inspect Results

Each completed seed produces a directory e.g.  `<welfare_function>_<nfe>_<seed>/` inside `results/` containing:
- `pareto_front_<seed>.csv` — the final Pareto-optimal solutions (levers + objectives).
- `<welfare_function>_<nfe>_<seed>.tar.gz` — the ArchiveLogger convergence history (used in Assignment 6).
- `convergence_<seed>.csv` — EpsilonProgress and operator probabilities per NFE checkpoint (used in Assignment 6).

Load all available Pareto front CSVs and print a statistical summary.  
>Tip: you can use print(all_results[OBJECTIVE_COLS].describe().round(n))
Print the nfes, and number of solutions in the pareto front as well. 

In [9]:
import glob
import os

# Path to results (assignments_ema/results/)
RESULTS_ROOT = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "results"))

# Objective and lever column names
OBJECTIVE_COLS = ['welfare', 'fraction_above_threshold', 'welfare_loss_damage', 'welfare_loss_abatement']

# Find all pareto_front CSVs across all seed subdirectories
csv_files = glob.glob(os.path.join(RESULTS_ROOT, "**", "pareto_front_*.csv"), recursive=True)

if not csv_files:
    print(f"No pareto_front_*.csv files found in {RESULTS_ROOT}")
else:
    # Load each CSV and tag with seed and NFE extracted from the folder name
    dfs = []
    for f in sorted(csv_files):
        df = pd.read_csv(f)
        folder = os.path.basename(os.path.dirname(f))   # e.g. utilitarian_2000_1
        parts  = folder.split("_")
        df['seed'] = parts[-1]       # last element is the seed
        df['nfe']  = parts[-2]       # second to last is the NFE
        dfs.append(df)

    all_results = pd.concat(dfs, ignore_index=True)

    print(f"Results root  : {RESULTS_ROOT}")
    print(f"CSV files found: {len(csv_files)}")
    print(f"NFE per seed  : {all_results['nfe'].unique()}")
    print(f"Seeds loaded  : {sorted(all_results['seed'].unique())}")
    print(f"Total solutions (all seeds combined): {len(all_results)}")
    print(f"Solutions per seed:")
    print(all_results.groupby('seed').size().to_string())
    print()
    print("Objective statistics across all seeds:")
    print(all_results[OBJECTIVE_COLS].describe().round(3).to_string())


Results root  : c:\Users\marij\OneDrive\Documenten\Master-EPA\Model-Based-DM\epa141a_01\assignments_ema\results
CSV files found: 5
NFE per seed  : ['2000']
Seeds loaded  : ['1', '2', '3', '4', '5']
Total solutions (all seeds combined): 43
Solutions per seed:
seed
1     8
2    12
3     9
4     6
5     8

Objective statistics across all seeds:
           welfare  fraction_above_threshold  welfare_loss_damage  welfare_loss_abatement
count       43.000                    43.000               43.000                  43.000
mean    116370.542                     0.665             3212.334               11447.758
std     324319.433                     0.326             1179.054                6404.755
min        103.447                     0.000                0.000                   0.000
25%        103.457                     0.500             3629.986               10852.223
50%        103.483                     0.700             3633.788               10974.152
75%        103.571        

In [10]:
# Verify results are non-trivial:
# A non-trivial Pareto front should have:
#   - More than 1 solution
#   - Some variation in all four objective columns
#   - fraction_above_threshold < 1.0 for at least some solutions
#     (meaning the MOEA found policies better than BAU)

n_solutions = len(all_results)
n_above_bau = (all_results['fraction_above_threshold'] >= 1.0).sum()
n_below_bau = (all_results['fraction_above_threshold'] <  1.0).sum()
variation   = all_results[OBJECTIVE_COLS].std()

print("Non-triviality checks:")
print(f"  Total solutions          : {n_solutions}  {'✓' if n_solutions > 1 else '✗ only 1 solution'}")
print()
print("  Objective variation (std):")
for col in OBJECTIVE_COLS:
    flag = '✓' if variation[col] > 0 else '✗ no variation'
    print(f"    {col:<35} std = {variation[col]:.3f}  {flag}")
print()
print(f"  Solutions with fraction_above_threshold < 1.0 : {n_below_bau}  "
      f"({'✓ MOEA improved on BAU' if n_below_bau > 0 else '✗ no improvement over BAU'})")
print(f"  Solutions with fraction_above_threshold >= 1.0: {n_above_bau}")

Non-triviality checks:
  Total solutions          : 43  ✓

  Objective variation (std):
    welfare                             std = 324319.433  ✓
    fraction_above_threshold            std = 0.326  ✓
    welfare_loss_damage                 std = 1179.054  ✓
    welfare_loss_abatement              std = 6404.755  ✓

  Solutions with fraction_above_threshold < 1.0 : 33  (✓ MOEA improved on BAU)
  Solutions with fraction_above_threshold >= 1.0: 10


---

## Reflection Questions

**1. Multiple seeds.** Why do we run the MOEA with 5 different random seeds rather than running a single long seed for 5× the NFE? What property of the algorithm makes this necessary?

To make sure the randomness is spread. One seed would mean one random configuration. By including multiple seeds, multiple random configurations are included in the results.

**2. NFE size.**  What diagnostic would tell you whether the NFE you tried was enough?

The hypervolume? But how to get it.

**3. Operator adaptation.** GenerationalBorg continuously adapts the selection probabilities of its six variation operators (SBX, PCX, DE, UNDX, SPX, UM) during the run. What does this mean and why is it useful for the 244-dimensional RBF search space?  What is a key tradeoff of this MOEA's adaptive learning feature?




### Claude's answer to reflection question 2: 
**Question: What diagnostic would tell you whether the NFE you tried was enough?**

The primary diagnostic is the **epsilon progress plot** — this is exactly what the `convergence_<seed>.csv` files produced by each run are for. Epsilon progress tracks how many times the epsilon-archive improves (i.e. a new non-dominated solution is added) as a function of NFE. 

If NFE was sufficient, the epsilon progress curve will **flatten out** well before the final NFE, meaning the archive stopped improving and the algorithm had converged. If the curve is still rising steeply at the end of the run, the algorithm was still finding improvements when it stopped — meaning more NFE would likely produce a better Pareto front.

A second supporting diagnostic is **seed-to-seed consistency**. If all 5 seeds produce Pareto fronts of similar size and objective range, the algorithm is consistently finding the same region of the objective space, which gives confidence that the front is stable. If the fronts differ substantially between seeds, the search has not converged and more NFE is needed.

A third check is the **hypervolume indicator** — the volume of objective space dominated by the combined Pareto front. Plotting hypervolume against NFE across seeds gives a convergence curve analogous to epsilon progress. A plateau indicates convergence; a still-rising curve indicates more evaluations are needed.

In practice for this assignment with `NFE=2000` and a 244-dimensional search space, convergence is unlikely to be fully achieved — 2000 evaluations is intentionally kept small for local runs. The honest answer is that you would need significantly more NFE (the script default is 50,000) to approach true convergence on this problem.

### Claude on question 3:

Why this is useful for the 244-dimensional RBF space
In a 244-dimensional search space the landscape is extremely complex — different regions may have very different local geometries. An operator that works well early in the search (when the archive is sparse and broad exploration is needed) may not work well later (when fine exploitation around good solutions is more valuable). No single operator is universally best across the entire search trajectory. Borg's adaptation means it automatically shifts strategy as the search progresses, without requiring you to manually tune operator selection probabilities beforehand.

The key tradeoff
The adaptation mechanism itself requires additional function evaluations to learn which operators are performing well. Early in the run, Borg must try all operators sufficiently to estimate their relative performance — this is essentially an exploration-exploitation tradeoff at the meta-level. With a limited NFE budget like 2000, a significant fraction of evaluations may be spent on this operator-learning phase rather than directly improving the Pareto front. A simpler single-operator algorithm like NSGA-II would spend all its evaluations on the actual search, potentially outperforming Borg at very low NFE budgets even if Borg would win at higher budgets where the adaptation pays off.

---

## Appendix — Script Reference

```
run_optimization_local.py — full argument list

  --nfe           int     NFE per seed                 (default: 50000)
  --seeds         int+    Random seeds                 (default: all 5)
  --output_dir    str     Results root directory        (default: answers_ema/results/)
  --n_processes   int     Worker processes              (default: auto)
  --n_ensembles   int     FAIR ensemble members         (default: 10)
  --config        str     JSON config path              (default: config/config_student.json)
  --population    int     MOEA population size          (default: 100)
```

**Output directory structure:**

```
assignments_ema/
└── results/
    └── <welfare_function>_<nfe>_<seed>/
        ├── <welfare_function>_<nfe>_<seed>.tar.gz   ← convergence archive (Assignment 6)
        ├── convergence_<seed>.csv            ← EpsilonProgress + operator probs (Assignment 6)
        └── pareto_front_<seed>.csv           ← final Pareto front
```

For example, running with `--nfe 50000 --seeds 9845531` produces:

```
assignments_ema/
└── results/
    └── <welfare_function>_50000_9845531/
        ├── <welfare_function>_50000_9845531.tar.gz
        ├── convergence_9845531.csv
        └── pareto_front_9845531.csv
```